# Imports

In [1]:
import boto3
from dotenv import load_dotenv
import os

## Using Client

In [3]:
load_dotenv(override=True)

True

In [2]:
s3 = boto3.client(
    "s3",
    endpoint_url="https://garage.learndatascience.cloud",
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    region_name="garage",
)
# https://realpython.com/python-boto3-aws-s3/

In [4]:
print(os.environ["AWS_ACCESS_KEY_ID"])
print(os.environ["AWS_SECRET_ACCESS_KEY"])

GKc09b7d0b9648439df630263f
12cb8fdb0878dd3abe634e3ec7706022a8a18afc7b3b3283c0070f5272f20988


In [5]:
# Print out bucket names
s3_ressources = boto3.resource("s3")
for bucket in s3_ressources.buckets.all():
    print(bucket.name)

# Ne marche pas

jenedai


In [55]:
config = load_dotenv()

## verifier le Bucket

In [ ]:
s3_client = boto3.client("s3")

try:
    response = s3_client.list_buckets()
    print("S3 Buckets:", response)
except Exception as e:
    print(f"An error occurred: {e}")

In [10]:
file_path = "../../Services/Garage_s3_FL/extract_cvs_engis_dataset_500000.csv"
bucket_name = "jenedai"
object_key = "dataset_train.csv"
from botocore.exceptions import NoCredentialsError, ClientError

try:
    # Upload the file
    s3.upload_file(file_path, bucket_name, object_key)
    print(f"File '{file_path}' uploaded successfully to S3 bucket '{bucket_name}/{object_key}'.")
except FileNotFoundError:
    print(f"The file {file_path} was not found.")
except NoCredentialsError:
    print("Credentials not available. Please verify your AWS credentials.")
except ClientError as e:
    error_code = e.response["Error"]["Code"]
    if error_code == "EntityTooLarge":
        print(f"Error: The file size exceeds the allowed limit for a single upload request.")
    else:
        print(f"Client Error {error_code}: {str(e)}")
except Exception as e:
    print("An unexpected error occurred:", str(e))

An unexpected error occurred: Failed to upload ../../Services/Garage_s3_FL/extract_cvs_engis_dataset_500000.csv to jenedai/dataset_train.csv: An error occurred (413) when calling the UploadPart operation: Request Entity Too Large


In [11]:
s3.upload_file?

Signature:
s3.upload_file(
    Filename,
    Bucket,
    Key,
    ExtraArgs=None,
    Callback=None,
    Config=None,
)
Docstring:
Upload a file to an S3 object.

Usage::

    import boto3
    s3 = boto3.client('s3')
    s3.upload_file('/tmp/hello.txt', 'amzn-s3-demo-bucket', 'hello.txt')

Similar behavior as S3Transfer's upload_file() method, except that
argument names are capitalized. Detailed examples can be found at
:ref:`S3Transfer's Usage <ref_s3transfer_usage>`.

:type Filename: str
:param Filename: The path to the file to upload.

:type Bucket: str
:param Bucket: The name of the bucket to upload to.

:type Key: str
:param Key: The name of the key to upload to.

:type ExtraArgs: dict
:param ExtraArgs: Extra arguments that may be passed to the
    client operation. For allowed upload arguments see
    :py:attr:`boto3.s3.transfer.S3Transfer.ALLOWED_UPLOAD_ARGS`.

:type Callback: function
:param Callback: A method which takes a number of bytes transferred to
    be periodically cal

In [18]:
from boto3.s3.transfer import TransferConfig

config = TransferConfig(
    multipart_threshold=5 * 1024 * 1024,  # 5 MB
    multipart_chunksize=5 * 1024 * 1024,  # 5 MB → ~18 parts pour 90 MB
    max_concurrency=2,
)
try:
    s3.upload_file(
        file_path,
        bucket_name,
        object_key,
        Config=config,
    )

    print(f"Successfully uploaded {file_path} to {bucket_name}/{object_key}")
except NoCredentialsError:
    print("Error: AWS credentials not found.")
except ClientError as e:
    print(f"AWS Client Error: {e.response['Error']['Message']}")
except Exception as e:
    print(f"Unexpected error: {e}")

Successfully uploaded ../../Services/Garage_s3_FL/extract_cvs_engis_dataset_500000.csv to jenedai/dataset_train.csv


### Lister les objets su le bucket

In [12]:
bucket_name = "jenedai"
# List objects in the bucket
response = s3.list_objects_v2(Bucket=bucket_name)

In [13]:
# Check if the bucket is not empty
if "Contents" in response:
    print(f"Files in bucket '{bucket_name}':")
    for obj in response["Contents"]:
        print(obj["Key"])
else:
    print(f"No files found in bucket '{bucket_name}'.")

Files in bucket 'jenedai':
dataset_train.csv
test.txt
test_s3.txt


### upload a file

In [ ]:
filepath = "../../Services/Garage_s3_FL/extract_cvs_engis_dataset_500000.csv"
s3.upload_file(filepath, "jenedai", "dataset_train.csv")

In [16]:
s3.meta.client.upload_file(Filename="fichier.txt", Bucket="jenedai", Key="fichier.txt")

In [3]:
import os

file_path = "../../Services/Garage_s3_FL/extract_cvs_engis_dataset_500000.csv"
print(f"File path: {file_path}")
print(f"File size: {os.path.getsize(file_path) / (1024 * 1024)} MB")

File path: ../../Services/Garage_s3_FL/extract_cvs_engis_dataset_500000.csv
File size: 90.72765350341797 MB


### Download a file

In [22]:
file_key = "dataset_train.csv"
local_path = "/tmp/dataset_train.csv"

# Télécharger le fichier
s3.download_file(bucket_name, object_key, local_path)

### Deleting a file

In [41]:
# Supprimer le fichier
s3.delete_object(Bucket=bucket_name, Key=file_key)

{'ResponseMetadata': {'HTTPStatusCode': 204,
  'HTTPHeaders': {'server': 'nginx',
   'date': 'Sun, 26 Apr 2026 12:12:09 GMT',
   'connection': 'keep-alive'},
  'RetryAttempts': 0}}

### Lister les objets su le bucket

In [42]:
response = s3.list_objects_v2(
    Bucket=bucket_name
    # Prefix='data/'
)
# Check if the bucket is not empty
if "Contents" in response:
    print(f"Files in bucket '{bucket_name}':")
    for obj in response["Contents"]:
        print(obj["Key"])
else:
    print(f"No files found in bucket '{bucket_name}'.")

Files in bucket 'jenedai':
test_s3.txt
